In [1]:
import pandas as pd
import re
from difflib import get_close_matches


In [2]:
# Read the genetic data
genetic_data = pd.read_csv(r"C:\Users\fatemehm\OneDrive - Royal HZPC Group\Desktop\internship\bio_rep1\GeneMarkers-95 varieties 1.csv")

genetic_data.info
genetic_data.head(10)

,taglo_id,ATLANTIC_124776,ESCORT_159228,ARGOS_120931,ALTURAS_329540,ELMUNDO_835520,NADINE_241950,VIOLETQUEEN_3345402,DEODARA_513721,MEMPHIS_2279529,...,DIAMANT_153502,INNOVATOR_234757,TRIPLE7_3347283,SPUNTA_283648,ANTI_118190,CARRERA_142554,FORTUS_3279866,SMART_1883446,SALINERO_253609,FABULA_171173
0,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,10,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,13,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,17,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,21,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,22,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,39,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,48,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,49,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,54,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


In [3]:
genetic_data.isnull().sum()

taglo_id           0
ATLANTIC_124776    0
ESCORT_159228      0
ARGOS_120931       0
ALTURAS_329540     0
                  ..
CARRERA_142554     0
FORTUS_3279866     0
SMART_1883446      0
SALINERO_253609    0
FABULA_171173      0
Length: 95, dtype: int64

In [4]:
# Clean column names
genetic_columns_clean = [c.strip().upper().replace(" ", "").replace(".", "") for c in genetic_data.columns]

# Assign cleaned names back to DataFrame
genetic_data.columns = genetic_columns_clean

# Now you can view columns
print(genetic_data.columns)


Index(['TAGLO_ID', 'ATLANTIC_124776', 'ESCORT_159228', 'ARGOS_120931',
       'ALTURAS_329540', 'ELMUNDO_835520', 'NADINE_241950',
       'VIOLETQUEEN_3345402', 'DEODARA_513721', 'MEMPHIS_2279529',
       'AGRIA_125336', 'PAYETTERUSSET_4617676', 'IVORYRUSSET_1360551',
       'SABABA_4959615', 'CECILE_147140', 'FRISIA_163592', 'CARDYMA_3345873',
       'RICKEYRUSSET_4640884', 'ANIVIA_4446779', 'AVARNA_1630052',
       'DONALD_158741', 'JAERLA_231308', 'ARIZONA_3056017',
       'SARPOMIRA_1629377', 'MARILYN_1632173', 'DESIREE_139717',
       'MUSE_6159099', 'ADORA_123802', 'TAURUS_1883495', 'BERBER_121293',
       'RANGERRUSSET_265488', 'BINTJE_126680', 'FENWAYRED_4055844',
       'KONDOR_248484', 'SAGITTA_1459411', 'CLEARWATERR_2721777',
       'NICOLA_248062', 'TETONRUSSET_3029162', 'FESTIEN_172619',
       'HANSA_180216', 'PEEWEERUSSET_3347291', 'PRINCEOFORANGE_416115',
       'SUPERIOR_1477892', 'VANGOGH_294181', 'CHALLENGER_1883438',
       'JENNIFER_3221728', 'PARELLA_2983716', 'CH

In [5]:
merged_aroma_sensory= pd.read_csv(r'C:\Users\fatemehm\OneDrive - Royal HZPC Group\Desktop\internship\bio_rep1\merged_aroma_sensory.csv')
print(merged_aroma_sensory)

         Variety  1-Nonanol (Floral, waxy)  \
0          ADORA                       0.0   
1          AGRIA                       0.0   
2       ALOUETTE                       0.0   
3         ALTHEA                       0.0   
4        ALTURAS                       0.0   
..           ...                       ...   
89  TETON RUSSET                       0.0   
90       TRIPLE7                       0.0   
91      VAN GOGH                       0.0   
92     VE 71-105                       0.0   
93  VIOLET QUEEN                       0.0   

    2,3-Butanedione (Butter, creamy, sweet)  \
0                                       0.0   
1                                       0.0   
2                                       0.0   
3                                       0.0   
4                                       0.0   
..                                      ...   
89                                      0.0   
90                                      0.0   
91                      

In [10]:
import re

def normalize_variety(name):
    if not isinstance(name, str):
        return ""
    name = name.upper()
    name = name.replace(".", "")
    name = re.sub(r'\bR\b', '', name)
    name = re.sub(r'RUSSET', '', name)
    name = name.replace(" ", "")
    name = name.strip()
    name = name.split('_')[0]
    return name

# Normalize genetic data index
genetic_data_t = genetic_data.set_index('TAGLO_ID').T
genetic_data_t.index = genetic_data_t.index.map(normalize_variety)

# Same corrections dictionary
name_corrections = {
    'CLEARWATERR': 'CLEARWATER',
    'ALVERSTONER': 'ALVERSTONE'
}

# Apply corrections to genetic data index as well
genetic_data_t.index = genetic_data_t.index.to_series().replace(name_corrections)

# Normalize aroma sensory data
merged_aroma_sensory['Variety_norm'] = merged_aroma_sensory['Variety'].apply(normalize_variety)
merged_aroma_sensory['Variety_norm'] = merged_aroma_sensory['Variety_norm'].replace(name_corrections)
merged_aroma_sensory = merged_aroma_sensory.set_index('Variety_norm')

# Now merge on corrected index
combined_df = merged_aroma_sensory.merge(genetic_data_t, left_index=True, right_index=True, how='inner')

print(f"Combined shape after correction: {combined_df.shape}")

# Check missing varieties again
aroma_varieties = set(merged_aroma_sensory.index)
genetic_varieties = set(genetic_data_t.index)

missing_in_genetic = aroma_varieties - genetic_varieties
missing_in_aroma = genetic_varieties - aroma_varieties

print(f"Varieties missing in genetic data: {missing_in_genetic}")
print(f"Varieties missing in aroma/sensory data: {missing_in_aroma}")
combined_df.head(5)

Combined shape after correction: (94, 262461)
Varieties missing in genetic data: set()
Varieties missing in aroma/sensory data: set()


,Variety,"1-Nonanol (Floral, waxy)","2,3-Butanedione (Butter, creamy, sweet)","Benzaldehyde (Almond, cherry, fruity)","Butanal, 3-methyl- (Fruity, malty, chocolate)","Cyclotrisiloxane, hexamethyl- (Chemical, silicone-like)","Decanal (Citrus, floral)","Dimethyl trisulfide (Garlic, onion, sulfurous)","Furfural (Sweet, almond, caramel)","Hexanal (Green, grassy)",...,564850,564851,564852,564853,564854,564857,564858,564859,564860,564861
ADORA,ADORA,0.0,0.0,17.984870,15.836435,17.417208,18.203338,0.00000,0.0,0.00000,...,2,4,3,1,0,0,0,0,0,2
AGRIA,AGRIA,0.0,0.0,14.859493,14.154847,15.947810,14.499810,17.21077,0.0,0.00000,...,0,4,1,0,0,0,0,0,0,0
ALOUETTE,ALOUETTE,0.0,0.0,15.205705,15.262178,14.678352,17.462371,0.00000,0.0,0.00000,...,3,4,3,2,0,0,0,0,0,3
ALTHEA,ALTHEA,0.0,0.0,19.535081,14.984406,17.704453,21.158234,0.00000,0.0,0.00000,...,0,4,0,0,0,0,0,0,0,0
ALTURAS,ALTURAS,0.0,0.0,20.293776,14.482897,14.257052,16.590508,0.00000,0.0,17.24572,...,1,4,2,1,0,0,0,0,1,1


# Preprocessing: 

Split features and targets

Normalize or standardize the genetic markers for modeling.

In [ ]:
flavor_cols = [
    'Sweet', 'Intensity of Flavour', 'Metallic Flavour', 'Bitter Flavour',
    'Earthy Flavour', 'Sour Flavour', 'Fresh Flavour', 'Sweet Flavour',
    'Root/ Vegetable Flavour', 'Farmyard (grass/hay) flavour',
    'Bitter Aftertaste', 'Sour Aftertaste', 'Sweet Aftertaste'
]
exclude_cols = flavor_cols + ['Variety'] + [  # add aroma compounds if you want to exclude them
    '1-Nonanol (Floral, waxy)', '2,3-Butanedione (Butter, creamy, sweet)', 'Benzaldehyde (Almond, cherry, fruity)',
    'Butanal, 3-methyl- (Fruity, malty, chocolate)', 'Cyclotrisiloxane, hexamethyl- (Chemical, silicone-like)',
    'Decanal (Citrus, floral)', 'Dimethyl trisulfide (Garlic, onion, sulfurous)', 'Furfural (Sweet, almond, caramel)',
    'Hexanal (Green, grassy)', 'Hexanoic acid (Fatty, cheesy, sweaty)', 'Methional (Cooked potato, sulfurous)', 
    'Octanal (Citrus, fruity)', 'Pentanal (Green, fatty, pungent)'
]

X = combined_df.drop(columns=exclude_cols, errors='ignore')
y = combined_df[flavor_cols]

print(f"X shape: {X.shape}, y shape: {y.shape}")


X shape: (94, 262434), y shape: (94, 13)


94 samples (varieties)

262,434 genetic marker features in X

13 flavor targets in y

# Dimensionality reduction with PCA

In [18]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
# from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=10)  # Reduce dimensionality to 50 principal components
X_pca = pca.fit_transform(X_scaled)

print(f"Reduced X shape: {X_pca.shape}")

Reduced X shape: (94, 10)


Model using PLS Regression with PCA components

In [19]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_score, KFold
import numpy as np

flavor_to_predict = 'Sweet'

mask = ~y[flavor_to_predict].isna()
X_train = X_pca[mask]
y_train = y.loc[mask, flavor_to_predict]

pls = PLSRegression(n_components=10)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(pls, X_train, y_train, cv=cv, scoring='r2')
print(f"{flavor_to_predict} CV R^2 with PCA features: {np.mean(scores):.3f} ± {np.std(scores):.3f}")


Sweet CV R^2 with PCA features: -0.915 ± 1.285


Lasso regression

In [ ]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_score, KFold
import numpy as np

flavor_cols = [
    'Sweet', 'Intensity of Flavour', 'Metallic Flavour', 'Bitter Flavour',
    'Earthy Flavour', 'Sour Flavour', 'Fresh Flavour', 'Sweet Flavour',
    'Root/ Vegetable Flavour', 'Farmyard (grass/hay) flavour',
    'Bitter Aftertaste', 'Sour Aftertaste', 'Sweet Aftertaste'
]

results = {}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
pls = PLSRegression(n_components=10)

for flavor in flavor_cols:
    # Filter out samples with missing target for this flavor
    mask = ~y[flavor].isna()
    X_filtered = X_scaled[mask]
    y_filtered = y.loc[mask, flavor]

    if len(y_filtered) < 10:
        print(f"Skipping {flavor}: not enough samples after filtering")
        continue

    scores = cross_val_score(pls, X_filtered, y_filtered, cv=cv, scoring='r2')
    results[flavor] = (scores.mean(), scores.std())
    print(f"{flavor} CV R^2: {scores.mean():.3f} ± {scores.std():.3f}")



Sweet CV R^2: 0.010 ± 0.248
Intensity of Flavour CV R^2: 0.043 ± 0.136
Metallic Flavour CV R^2: -0.032 ± 0.148
Bitter Flavour CV R^2: -0.072 ± 0.150
Earthy Flavour CV R^2: -0.006 ± 0.105
Sour Flavour CV R^2: -0.016 ± 0.108
Fresh Flavour CV R^2: 0.095 ± 0.126
Sweet Flavour CV R^2: 0.069 ± 0.355
Root/ Vegetable Flavour CV R^2: 0.065 ± 0.150
Farmyard (grass/hay) flavour CV R^2: -0.054 ± 0.186
Bitter Aftertaste CV R^2: -0.068 ± 0.110
Sour Aftertaste CV R^2: -0.009 ± 0.112
Sweet Aftertaste CV R^2: -0.038 ± 0.495

Summary CV R² scores:
Sweet: 0.010 ± 0.248
Intensity of Flavour: 0.043 ± 0.136
Metallic Flavour: -0.032 ± 0.148
Bitter Flavour: -0.072 ± 0.150
Earthy Flavour: -0.006 ± 0.105
Sour Flavour: -0.016 ± 0.108
Fresh Flavour: 0.095 ± 0.126
Sweet Flavour: 0.069 ± 0.355
Root/ Vegetable Flavour: 0.065 ± 0.150
Farmyard (grass/hay) flavour: -0.054 ± 0.186
Bitter Aftertaste: -0.068 ± 0.110
Sour Aftertaste: -0.009 ± 0.112
Sweet Aftertaste: -0.038 ± 0.495


Filter genetic markers by variance

In [22]:
from sklearn.feature_selection import VarianceThreshold

# Keep markers with variance > 0.01 (adjust threshold as needed)
selector = VarianceThreshold(threshold=0.01)
X_var = selector.fit_transform(X_scaled)

print(f"Reduced markers: {X_var.shape[1]} (from {X_scaled.shape[1]})")


Reduced markers: 261110 (from 262434)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNetCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

# Flavor and aroma columns - adjust as needed
flavor_cols = [
    'Sweet', 'Intensity of Flavour', 'Metallic Flavour', 'Bitter Flavour',
    'Earthy Flavour', 'Sour Flavour', 'Fresh Flavour', 'Sweet Flavour',
    'Root/ Vegetable Flavour', 'Farmyard (grass/hay) flavour',
    'Bitter Aftertaste', 'Sour Aftertaste', 'Sweet Aftertaste'
]

aroma_cols = [
    '1-Nonanol (Floral, waxy)', '2,3-Butanedione (Butter, creamy, sweet)', 'Benzaldehyde (Almond, cherry, fruity)',
    'Butanal, 3-methyl- (Fruity, malty, chocolate)', 'Cyclotrisiloxane, hexamethyl- (Chemical, silicone-like)',
    'Decanal (Citrus, floral)', 'Dimethyl trisulfide (Garlic, onion, sulfurous)', 'Furfural (Sweet, almond, caramel)',
    'Hexanal (Green, grassy)', 'Hexanoic acid (Fatty, cheesy, sweaty)', 'Methional (Cooked potato, sulfurous)',
    'Octanal (Citrus, fruity)', 'Pentanal (Green, fatty, pungent)'
]

# Step 1: Variance threshold filtering on genetic markers
exclude_cols = flavor_cols + ['Variety'] + aroma_cols
X_genetic = combined_df.drop(columns=exclude_cols, errors='ignore')
y = combined_df[flavor_cols]
X_genetic = X_genetic.fillna(0)  # Fill NA in markers if any

selector = VarianceThreshold(threshold=0.01)
X_var = selector.fit_transform(X_genetic)

print(f"Reduced markers: {X_var.shape[1]} (from {X_genetic.shape[1]})")

# Step 2: Prepare aroma features and combine
X_aroma = combined_df[aroma_cols].fillna(0).values
X_combined = np.hstack([X_var, X_aroma])

print(f"Combined features shape: {X_combined.shape}")

# Step 3: Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_combined)

# Cross-validation setup
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Containers for results
results_enet = {}
results_rf = {}

# Step 4: Run Elastic Net and Random Forest for each flavor trait
for flavor in flavor_cols:
    mask = ~y[flavor].isna()
    if mask.sum() < 10:
        print(f"Skipping {flavor} due to too few samples")
        continue
    X_train = X_scaled[mask]
    y_train = y.loc[mask, flavor].values

    # Elastic Net
    enet = ElasticNetCV(cv=5, random_state=42)
    scores_enet = cross_val_score(enet, X_train, y_train, cv=cv, scoring='r2')
    results_enet[flavor] = (scores_enet.mean(), scores_enet.std())
    
    # Random Forest
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    scores_rf = cross_val_score(rf, X_train, y_train, cv=cv, scoring='r2')
    results_rf[flavor] = (scores_rf.mean(), scores_rf.std())
    
    print(f"{flavor}: ElasticNet R² = {scores_enet.mean():.3f} ± {scores_enet.std():.3f} | RandomForest R² = {scores_rf.mean():.3f} ± {scores_rf.std():.3f}")

# Step 5: PCA on filtered genetic markers for correlation heatmap
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X_var)

# Calculate correlation between PCs and flavor traits (only samples with all flavors present)
flavor_data = y.dropna(subset=flavor_cols)
pc_flavor_corr = np.corrcoef(X_pca.T, flavor_data.T)[:10, 10:]

# Plot correlation heatmap
plt.figure(figsize=(12,6))
sns.heatmap(pc_flavor_corr, xticklabels=flavor_cols, yticklabels=[f'PC{i+1}' for i in range(10)], cmap='coolwarm', center=0)
plt.title("Correlation between Genetic PCs and Flavor Traits")
plt.show()

# Step 6: Clustering varieties by combined features
kmeans = KMeans(n_clusters=4, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

combined_df['Cluster'] = np.nan
combined_df.loc[flavor_data.index, 'Cluster'] = clusters

print("\nCluster assignments for first 10 varieties:")
print(combined_df[['Variety', 'Cluster']].head(10))


Reduced markers: 261110 (from 262434)
Combined features shape: (94, 261123)
